# Notebook 3: Transcribe the test pages

This uses a trained run from Notebook 2 to read all 141 Kade test pages, line by line, and saves
the result in the format Kaggle wants (`page_id`, `text`). It first re-reads the 11 dev pages as a
check: their score should match what Notebook 2 reported.

**Needs a GPU**, like Notebook 2. Run the cells from top to bottom with **Shift + Enter**.
Kernel: **conda_pytorch_p310** (or whichever `conda_pytorch` kernel your instance lists).

## Step 1. Install the libraries

The same as Notebook 2, Step 1: needed after every start, and before any other cell.

In [ ]:
%pip install -q -r requirements-training.txt

## Step 2. Set up the folders

In [ ]:
import os
import sys

# Everything this project downloads or creates lives here. It is on the
# notebook instance's storage volume, so it survives Stop and Start.
WORK = "/home/ec2-user/SageMaker/badger-finetune"
DATA = os.path.join(WORK, "data")            # page images, labels, line images
RUNS = os.path.join(WORK, "runs")            # one folder per training run
SCRIPTS = os.path.abspath("scripts")         # the scripts folder next to this notebook
for folder in (WORK, DATA, RUNS):
    os.makedirs(folder, exist_ok=True)
sys.path.insert(0, SCRIPTS)

# Keep the 9 GB model download on the big volume, not the small system disk.
os.environ["HF_HOME"] = os.path.join(WORK, "hf_cache")

SHARED_BUCKET = "ml-marathon-shared"   # the class bucket with the Badger Scribe data
SHARED_PREFIX = "badger-scribe"
print("Working folder:", WORK)

## Step 3. Pick the run to use

This lists your runs from best to worst dev score. The best one is picked automatically; to use a
different one, type its name. Smoke runs are left out: they only score 2 dev pages, so their
score can't be compared with the others. `RUN_NAME = None` means no adapter of ours: just the Kurrent adapter
as it was before training (useful for comparison).

In [ ]:
import json
import pandas as pd

runs = []
for name in os.listdir(RUNS):
    path = os.path.join(RUNS, name, "model", "train_summary.json")
    if os.path.exists(path) and not name.startswith("smoke"):
        s = json.load(open(path))
        runs.append({"run": name, "dev CER (best)": s["dev_cer_best"], "before training": s["dev_cer_before"],
                     "dev pages": s["dev_pages"]})
runs = pd.DataFrame(runs).sort_values("dev CER (best)") if runs else pd.DataFrame()
print(runs.to_string(index=False) if len(runs) else "No finished baseline or full runs yet: run Notebook 2 first.")

RUN_NAME = runs.iloc[0]["run"] if len(runs) else None   # or type a run name, e.g. "full-lr0.0001-..."
adapter_dir = os.path.join(RUNS, RUN_NAME, "model") if RUN_NAME else None
label = RUN_NAME or "kurrent-adapter-only"
print("\nUsing:", label)

## Step 4. Check on the dev pages

Re-reads the 11 dev pages, whose correct text we know, and scores them. The average should be the
same as the run's best score in Notebook 2. If it's very different, something doesn't match
between training and reading, and the test predictions shouldn't be trusted. (About 5 minutes.)

In [ ]:
out_dir = os.path.join(WORK, "predictions", label)
os.makedirs(out_dir, exist_ok=True)
lines_dir = os.path.join(DATA, "kade_lines")
python = sys.executable
adapter = f"--adapter_dir {adapter_dir}" if adapter_dir else ""

dev_out = os.path.join(out_dir, "dev_check.csv")
!{python} {SCRIPTS}/transcribe_pages.py --lines_dir {lines_dir} --pages {DATA}/dev_pages.csv --out {dev_out} {adapter}
print(f"\nMean dev CER: {pd.read_csv(dev_out, keep_default_na=False)['cer'].mean():.3f}")

## Step 5. Transcribe the test pages

Reads every page in `test.csv` (about 15–25 minutes). If it stops partway, run the cell again: it
continues where it stopped.

In [ ]:
test_out = os.path.join(out_dir, "test_raw.csv")
test_pages = pd.read_csv(os.path.join(DATA, "test.csv"), keep_default_na=False)[["page_id"]]
test_pages.to_csv(os.path.join(out_dir, "test_pages.csv"), index=False)
!{python} {SCRIPTS}/transcribe_pages.py --lines_dir {lines_dir} --pages {out_dir}/test_pages.csv --out {test_out} {adapter}

## Step 6. Save the predictions in Kaggle's format

This puts the pages in the same order as `test.csv`, with exactly the two columns Kaggle accepts,
and checks that no page is missing or empty.

In [ ]:
preds = pd.read_csv(test_out, keep_default_na=False)
submission = test_pages.merge(preds[["page_id", "text"]], on="page_id", how="left").fillna({"text": ""})
empty = submission[submission.text.str.strip() == ""]
if len(empty):
    print(f"WARNING: {len(empty)} pages have no text: {', '.join(empty.page_id[:10])}")
path = os.path.join(out_dir, "kade_predictions.csv")
submission.to_csv(path, index=False)
print(f"{len(submission)} pages saved to {path}")
submission.head()

## Step 7. Download and share

- **To download** the file to your computer: in the file browser on the left, go to
  `badger-finetune/predictions/<run>/`, right-click `kade_predictions.csv`, and choose
  **Download**. (On a Mac it lands in Downloads, like any browser download.)
- **To share it with the team**, copy it to your bucket with the cell below.

These are predictions for the **Kade pages only**. If the team's Kaggle submission also has
other kinds of pages (surveys, account books), replace just the Kade rows of that file with these.

Then **stop the instance** if you're done (README.md, "Stopping and starting").

In [ ]:
MY_BUCKET = "your-bucket-name"   # <-- your own bucket

!aws s3 sync {out_dir} s3://{MY_BUCKET}/badger-finetune/predictions/{label} --quiet
print(f"Saved to s3://{MY_BUCKET}/badger-finetune/predictions/{label}")